<a id="comparison"></a>
# Comparaison finale — mot-entier vs SentencePiece, sur données réelles, tagset fin

Les deux modèles ont été entraînés et évalués sur exactement le même split de CorBaMa (seed=42), donc la comparaison est équitable.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from bambara_pos_utils import *

import torch
from sklearn.metrics import accuracy_score, classification_report
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

corbama_fine_data = load_bambara_corpus(Path.cwd().parent / "08_real_data_evaluation" / "corbama_corpus_fine.tsv")
label2id_fine, id2label_fine = build_fine_tagset(corbama_fine_data)
corbama_train, corbama_val, corbama_test = split_corpus(corbama_fine_data, seed=42)

Corpus chargé : 166361 phrases.


In [ ]:
# Rechargement des deux modèles déjà entraînés (pas de réentraînement ici)
word_to_ix_cb, _ = build_vocab(corbama_train)   # même train split -> même vocabulaire reconstruit

model_word = load_checkpoint_generic(
    Path.cwd() / "models" / "corbama_word_finetagset.pth", device
)
model_sp = load_checkpoint_generic(
    Path.cwd() / "models" / "corbama_sp_finetagset.pth", device
)
sp_cb = spm.SentencePieceProcessor(model_file=str(Path.cwd() / "corbama_sp.model"))

print("Modèles rechargés.")

KeyError: 'vocab_size'

In [ ]:
def evaluate_word_model_fine(model, word_to_ix, sentences, id2label, device):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for words, tags in sentences:
            ids = [word_to_ix.get(w, word_to_ix["<UNK>"]) for w in words]
            input_tensor = torch.tensor([ids], dtype=torch.long).to(device)
            preds = torch.argmax(model(input_tensor), dim=-1).squeeze(0).cpu().tolist()
            all_true.extend(tags)
            all_pred.extend([id2label[p] for p in preds])
    return all_true, all_pred

true_word, pred_word = evaluate_word_model_fine(model_word, word_to_ix_cb, corbama_test, id2label_fine, device)
acc_word = accuracy_score(true_word, pred_word)

acc_sp, report_sp = evaluate_subword_model_word_level(model_sp, corbama_test, sp_cb, id2label_fine, "broadcast", device)

# taux de mots inconnus, cote mot-entier
unk_count = sum(1 for words, _ in corbama_test for w in words if w not in word_to_ix_cb)
total_tokens = sum(len(words) for words, _ in corbama_test)

print(f"{'Approche':<25}{'Accuracy':<15}{'Notes'}")
print(f"{'Mot-entier':<25}{acc_word*100:.2f}%{'':<10}Taux <UNK> = {100*unk_count/total_tokens:.1f}%")
print(f"{'SentencePiece':<25}{acc_sp*100:.2f}%")

NameError: name 'model_word' is not defined

In [ ]:
print("\n--- Rapport détaillé SentencePiece ---")
print(report_sp)
print("\n--- Rapport détaillé mot-entier ---")
print(classification_report(true_word, pred_word, zero_division=0))

# Interprétation 

Sur un corpus aussi petit (3711 phrases d'entraînement), le taux de mots inconnus côté mot-entier est probablement significatif sur le test set — c'est exactement le problème que SentencePiece est censé atténuer. Si SentencePiece gagne ici, c'est une confirmation cohérente avec le module 09. Si les deux performent mal sur les catégories rares du tagset fin (onomat, intj...), la conclusion honnête est que 24 catégories sur un corpus de 4 640 phrases, c'est probablement trop fin pour la quantité de données disponible — 